In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from collections import deque
from collections import defaultdict
import os
from tqdm import tqdm
from utlis import *
%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'utlis'

In [ ]:
# 1. Pattern filter — removes non-semantic categories matching patterns like Wikipedia_*, *_stubs, List_of_*, *_by_country, *_people, etc. 
# (administrative, maintenance, list-type categories)

# 2. BFS from root — starting at Main_topic_classifications, does a breadth-first traversal of the parent→child hierarchy and 
# keeps only categories reachable from that root

# 3. Remove isolated leaves (iterative) — repeatedly prunes leaf categories that have fewer than min_pages_for_leaf=5 articles, 
# until no more can be removed (cascading: removing a leaf can expose new leaves)

# 4. Filter page-category links — keeps only (page_id, category_id) rows where the category survived steps 1–3

# 5. Save to /cephfs/liuxinyu/wikiKG/tests/semantic_category/:

# semantic_categories.parquet
# semantic_category_edges.parquet
# semantic_page_categories.parquet

# Return value: reachable = the set of category_ids that made it into the final subgraph.

reachable = extract_semantic_subgraph(
    "/cephfs/liuxinyu/wikiKG/data/graph/category",
    "/cephfs/liuxinyu/wikiKG/WorldKnowledgeGraph/",
    root_title="Main_topic_classifications",
    apply_pattern_filter=True,
    remove_isolated=True,
    min_pages_for_leaf=5,
)